In [ ]:
import requests
import pandas as pd
import time
import os

# 1. API Bilgileri
API_KEY = 'c48580a258c9f1d82e2e2429b64e02ff' # Buraya kendi TMDB API anahtarını gir
BASE_URL = 'https://api.themoviedb.org/3'

# 2. Veri Çekme Fonksiyonu
def get_movie_data(movie_name):
    # Film isminin boş veya geçersiz olup olmadığını kontrol et
    if pd.isna(movie_name) or str(movie_name).strip() == "":
        return None

    try:
        # Arama Adımı (Film ID'sini bulmak için)
        search_url = f"{BASE_URL}/search/movie"
        params = {
            'api_key': API_KEY,
            'query': str(movie_name).strip()
        }

        response = requests.get(search_url, params=params)
        response.raise_for_status() # HTTP hatalarını yakala
        results = response.json().get('results')

        if not results:
            print(f"Uyarı: '{movie_name}' için sonuç bulunamadı.")
            return {
                'Film Adı': movie_name,
                'Bütçe': None,
                'Kazanç': None,
                'ID': None,
                'Durum': 'Bulunamadı'
            }

        # En alakalı (ilk) sonucu alıyoruz
        movie_id = results[0]['id']

        # Detay Adımı (Bütçe ve kazanç verilerini çekmek için)
        detail_url = f"{BASE_URL}/movie/{movie_id}"
        detail_params = {'api_key': API_KEY}

        detail_response = requests.get(detail_url, params=detail_params)
        detail_response.raise_for_status()
        data = detail_response.json()

        return {
            'Film Adı': movie_name,
            'Bütçe': data.get('budget'),
            'Kazanç': data.get('revenue'),
            'ID': movie_id,
            'Durum': 'Başarılı'
        }

    except Exception as e:
        print(f"Hata: '{movie_name}' işlenirken bir sorun oluştu. Hata detayı: {e}")
        return {
            'Film Adı': movie_name,
            'Bütçe': None,
            'Kazanç': None,
            'ID': None,
            'Durum': f'Hata: {e}'
        }

# 3. CSV'den Veri Okuma ve İşleme Ana Bloğu
def main():
    girdi_dosyasi = 'film_listesi.csv'   # Okunacak CSV dosyasının yolu
    cikti_dosyasi = 'film_verileri_sonuc.csv' # Kaydedilecek CSV dosyasının yolu
    sutun_adi = 'film_adi'               # Film isimlerinin olduğu sütunun başlığı

    # Dosyanın var olup olmadığını kontrol et
    if not os.path.exists(girdi_dosyasi):
        print(f"Hata: '{girdi_dosyasi}' adlı dosya bulunamadı. Lütfen dosya yolunu kontrol edin.")
        return

    # CSV'yi Pandas ile oku
    print(f"{girdi_dosyasi} okunuyor...")
    df_girdi = pd.read_csv(girdi_dosyasi)

    # Sütun kontrolü
    if sutun_adi not in df_girdi.columns:
        print(f"Hata: CSV dosyasında '{sutun_adi}' adında bir sütun bulunamadı.")
        print(f"Mevcut sütunlar: {list(df_girdi.columns)}")
        return

    sonuclar = []
    toplam_film = len(df_girdi)

    print(f"Toplam {toplam_film} film işlenecek. Başlıyor...\n")

    # Listedeki her bir film için döngü
    for index, row in df_girdi.iterrows():
        film = row[sutun_adi]
        print(f"[{index + 1}/{toplam_film}] İşleniyor: {film}")

        veri = get_movie_data(film)
        if veri:
            sonuclar.append(veri)

        # API sınırlarına takılmamak için bekleme (Rate limit)
        time.sleep(0.2)

    # 4. Sonuçları Kaydetme
    print("\nVeri çekme işlemi tamamlandı. Sonuçlar kaydediliyor...")
    df_sonuc = pd.DataFrame(sonuclar)

    # Yeni bir CSV dosyasına aktar
    df_sonuc.to_csv(cikti_dosyasi, index=False, encoding='utf-8')
    print(f"İşlem başarılı! Veriler '{cikti_dosyasi}' dosyasına kaydedildi.")

# Kodu çalıştır
if __name__ == "__main__":
    main()

film_listesi.csv okunuyor...
Toplam 49 film işlenecek. Başlıyor...

[1/49] İşleniyor: The Fall Guy 2024
Uyarı: 'The Fall Guy 2024' için sonuç bulunamadı.
[2/49] İşleniyor: Argylle
[3/49] İşleniyor: Land of Bad
[4/49] İşleniyor: The Beekeeper 2024
Uyarı: 'The Beekeeper 2024' için sonuç bulunamadı.
[5/49] İşleniyor: Rebel Ridge
[6/49] İşleniyor: Monkey Man 2024
Uyarı: 'Monkey Man 2024' için sonuç bulunamadı.
[7/49] İşleniyor: The Ministry of Ungentlemanly Warfare
[8/49] İşleniyor: Silent Night 2023
Uyarı: 'Silent Night 2023' için sonuç bulunamadı.
[9/49] İşleniyor: Kandahar
[10/49] İşleniyor: Plane 2023
Uyarı: 'Plane 2023' için sonuç bulunamadı.
[11/49] İşleniyor: Heart of Stone Gal Gadot
Uyarı: 'Heart of Stone Gal Gadot' için sonuç bulunamadı.
[12/49] İşleniyor: The Killer David Fincher
Uyarı: 'The Killer David Fincher' için sonuç bulunamadı.
[13/49] İşleniyor: Lift 2024
Uyarı: 'Lift 2024' için sonuç bulunamadı.
[14/49] İşleniyor: Manodrome
[15/49] İşleniyor: Boy Kills World
[16/49] İşl